<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **NOTEBOOK 21**
---

# **Técnicas de entrenamiento de modelos de lenguaje**

Un modelo de lenguaje moderno no se entrena de una sola vez, sino por **etapas**. Cada etapa utiliza un tipo de dato distinto y una señal de aprendizaje distinta, y aporta al modelo algo que la anterior no puede darle:

| Etapa | Datos | Señal de aprendizaje | Qué aporta |
|---|---|---|---|
| **1. Preentrenamiento autorregresivo** | Texto sin etiquetar, en cantidades enormes | Predecir el siguiente token | Conocimiento del lenguaje y del mundo |
| **2. Ajuste fino supervisado (SFT)** | Pares instrucción–respuesta | Imitar la respuesta de referencia | Comportamiento de asistente, formato, tarea o dominio concreto |
| **3. Aprendizaje por refuerzo (RL)** | Respuestas del propio modelo, puntuadas | Maximizar una recompensa | Cualidades que son fáciles de evaluar pero difíciles de demostrar con ejemplos |

Para adaptar un modelo a un **dominio específico** (jurídico, médico, educativo...) se reutilizan exactamente estas mismas técnicas, pero partiendo de un modelo ya preentrenado y con datos del dominio. En este notebook veremos con detalle las dos primeras, que son las que utilizaremos en la práctica, y comentaremos brevemente la tercera.

## **1. Entrenamiento autorregresivo**

### La idea

Un modelo autorregresivo descompone la probabilidad de una secuencia de tokens $x_1, \dots, x_T$ en un producto de probabilidades condicionadas, cada una de ellas dependiente solo de los tokens anteriores:

$$
P(x_1, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \dots, x_{t-1})
$$

Entrenar el modelo consiste en ajustar sus parámetros $\theta$ para que asigne la mayor probabilidad posible al token que realmente aparece a continuación en el texto. La función de pérdida es la **entropía cruzada** entre la distribución que predice el modelo y el token real (la misma que vimos en el tema 2), promediada sobre todas las posiciones:

$$
\mathcal{L}_{\text{AR}}(\theta) = -\frac{1}{T} \sum_{t=1}^{T} \log P_\theta(x_t \mid x_{<t})
$$

La **perplejidad**, $e^{\mathcal{L}_{\text{AR}}}$, es la forma habitual de comunicar esta pérdida: puede leerse como el número medio de tokens entre los que el modelo "duda" en cada paso.

### Aprendizaje autosupervisado

No hace falta que nadie etiquete los datos: la etiqueta de cada posición es, sencillamente, el token siguiente del propio texto. De una sola frase se obtienen tantos ejemplos de entrenamiento como tokens tiene:

| Contexto (entrada) | Token a predecir |
|---|---|
| `El` | `gato` |
| `El gato` | `duerme` |
| `El gato duerme` | `en` |
| `El gato duerme en` | `el` |
| `El gato duerme en el` | `sofá` |

Esto es lo que permite entrenar con cualquier texto disponible (libros, páginas web, código...) y, por tanto, a una escala que sería imposible con datos anotados a mano.

### Cómo se entrena en la práctica

- **Todas las posiciones a la vez.** Gracias a la **máscara causal** del Transformer, cada posición solo puede atender a las anteriores. Así, con una única pasada por la red se obtienen las $T$ predicciones de la secuencia y se calcula la pérdida de todas ellas en paralelo.
- **Teacher forcing.** Durante el entrenamiento, el contexto que recibe el modelo es siempre el texto real, no lo que él mismo habría generado. En inferencia ocurre lo contrario: cada token generado se añade al contexto y se vuelve a llamar al modelo. Esta diferencia entre entrenamiento e inferencia explica que un error temprano en la generación pueda arrastrarse al resto del texto.

### Qué aprende el modelo

El objetivo parece modesto, pero predecir bien el siguiente token obliga a modelar todo aquello de lo que depende el texto: la sintaxis, el significado, el estilo, hechos del mundo e incluso cadenas de razonamiento. Para continuar correctamente «La capital de Francia es», hay que saber geografía; para continuar una demostración, hay que seguir el argumento.

El resultado de esta etapa es un **modelo base**: un excelente *continuador de texto*, pero no un asistente. Ante el prompt «¿Cuál es la capital de Francia?», un modelo base puede responder, pero también puede continuar con más preguntas de examen, porque eso es igualmente una continuación plausible.

### Uso en dominios específicos: preentrenamiento continuado

Si disponemos de mucho texto de un dominio (sentencias judiciales, historias clínicas, manuales técnicos) pero no de ejemplos de preguntas y respuestas, podemos seguir entrenando un modelo ya preentrenado con el mismo objetivo autorregresivo sobre ese texto. Es lo que se conoce como *continued pretraining* o *domain-adaptive pretraining*.

- **Ventaja:** no requiere etiquetar nada. El modelo absorbe el vocabulario, el estilo y parte del conocimiento del dominio.
- **Limitación:** el modelo aprende a *escribir como* los documentos del dominio, no a *responder preguntas* sobre él. Normalmente va seguido de una etapa de SFT.

## **2. Ajuste fino supervisado (*Supervised Fine-Tuning*, SFT)**

### La idea

El SFT parte de un modelo preentrenado y lo entrena con ejemplos de **cómo queremos que se comporte**: pares formados por una entrada (instrucción, pregunta, conversación) y la salida deseada.

Lo más importante que hay que entender es que **el mecanismo de aprendizaje no cambia**: se sigue prediciendo el siguiente token y se sigue minimizando la entropía cruzada. Lo que cambia son los datos. Cada ejemplo se convierte en una única secuencia de texto mediante una **plantilla de chat** que marca, con tokens especiales, dónde empieza y termina cada turno. En el formato de Mistral, por ejemplo:

```
<s>[INST] ¿Cuál es la capital de Francia? [/INST] La capital de Francia es París. </s>
```

### La pérdida

Habitualmente la pérdida se calcula **solo sobre los tokens de la respuesta**. Los tokens de la instrucción forman parte del contexto, pero no se le pide al modelo que aprenda a generarlos:

$$
\mathcal{L}_{\text{SFT}}(\theta) = -\sum_{t \,\in\, \text{respuesta}} \log P_\theta(y_t \mid \text{instrucción},\, y_{<t})
$$

En la práctica esto se implementa enmascarando las etiquetas de las posiciones de la instrucción para que no contribuyan a la pérdida. (Algunas implementaciones sencillas calculan la pérdida sobre la secuencia completa; también funciona, aunque dedica parte de la capacidad a aprender a predecir instrucciones.)

### Qué aprende el modelo

- A adoptar el **papel de asistente**: lo que sigue a `[/INST]` es una respuesta, no más preguntas.
- A **terminar**: el token de fin de secuencia `</s>` forma parte de la respuesta, de modo que el modelo aprende cuándo parar.
- El **formato, el tono y las convenciones** de las respuestas del conjunto de datos.
- La **tarea o el dominio** concreto que cubren los ejemplos.

Como la plantilla es parte de lo aprendido, en inferencia hay que utilizar exactamente la misma con la que se entrenó. Un prompt con otro formato queda fuera de la distribución de entrenamiento y el rendimiento cae.

### Los datos

En SFT la calidad de los datos importa más que la cantidad: con unos pocos miles de ejemplos buenos se consiguen cambios de comportamiento notables. Las fuentes habituales son:

- ejemplos escritos o revisados por personas expertas en el dominio;
- conjuntos de datos existentes, reconvertidos al formato instrucción–respuesta;
- ejemplos **sintéticos**, generados por un modelo más capaz y filtrados después (una forma de destilación).

### Limitaciones

- **Es aprendizaje por imitación.** El modelo no puede ser mejor que los ejemplos que imita, y hereda sus errores y sesgos.
- **Una sola respuesta "correcta".** La pérdida premia reproducir la respuesta de referencia token a token, y penaliza otras respuestas igual de válidas pero redactadas de otra manera.
- **Sobreajuste y olvido catastrófico.** Con pocos datos y muchos parámetros, el modelo puede memorizar los ejemplos y degradar capacidades que tenía. Se mitiga con tasas de aprendizaje pequeñas, pocas épocas y, sobre todo, con las técnicas PEFT del siguiente notebook.
- **Alucinaciones.** Si los ejemplos contienen respuestas sobre hechos que el modelo no conoce, lo que aprende no es el hecho, sino a responder con seguridad aunque no sepa.

### Autorregresivo frente a SFT

| | Entrenamiento autorregresivo | SFT |
|---|---|---|
| **Datos** | Texto plano | Pares instrucción–respuesta con plantilla |
| **Etiquetas** | El propio texto (autosupervisado) | Respuestas de referencia (supervisado) |
| **Pérdida** | Entropía cruzada en todos los tokens | Entropía cruzada en los tokens de la respuesta |
| **Volumen de datos** | Muy grande | De miles a millones de ejemplos |
| **Resultado** | Modelo base: continúa texto | Modelo *instruct* o de chat: sigue instrucciones |
| **En un dominio específico** | Aporta vocabulario, estilo y conocimiento | Aporta la tarea y la forma de responder |

## **3. Aprendizaje por refuerzo: una nota breve**

El SFT enseña por imitación, y eso tiene un techo. Hay cualidades que son **mucho más fáciles de evaluar que de demostrar**: es sencillo decir cuál de dos respuestas es más útil, o comprobar si el resultado de un problema es correcto, pero es muy difícil escribir "la" respuesta ideal para cada caso. El aprendizaje por refuerzo aprovecha justamente eso.

El esquema es el siguiente: el modelo (la *política*, $\pi_\theta$) **genera** sus propias respuestas, una función de recompensa $R$ las **puntúa** y los parámetros se actualizan para que las respuestas mejor puntuadas sean más probables:

$$
J(\theta) = \mathbb{E}_{\,y \sim \pi_\theta(\cdot \mid x)}\big[R(x, y)\big]
$$

La diferencia con el SFT es clara: el SFT aumenta la probabilidad de una respuesta *que le damos*; el RL aumenta la probabilidad de las respuestas *que el propio modelo ha producido*, en proporción a la recompensa que han recibido. Por eso el modelo puede llegar a superar a cualquier ejemplo de referencia.

Según de dónde venga la recompensa, se distinguen dos usos principales:

- **Preferencias humanas (RLHF).** Se pide a personas que comparen pares de respuestas, se entrena con esas comparaciones un **modelo de recompensa** que predice la preferencia humana, y se optimiza el LLM contra ese modelo (habitualmente con PPO), añadiendo una penalización que impide que se aleje demasiado del modelo SFT de partida. Es la técnica que hay detrás de InstructGPT y de los primeros ChatGPT. Una alternativa muy utilizada es **DPO**, que aprovecha las mismas comparaciones con una pérdida de tipo supervisado, sin modelo de recompensa ni bucle de RL.
- **Recompensas verificables.** En matemáticas o programación la recompensa puede calcularse automáticamente: el resultado coincide o no, los tests pasan o no. Así se entrenan los **modelos de razonamiento**, que aprenden por sí mismos cadenas de pensamiento largas. Lo veremos en el tema 9 con [DeepSeek R1](https://nbviewer.org/urls/cayetanoguerra.github.io/ia/nbpy/nlp/Tema-09/nb02.ipynb).

El RL es más costoso e inestable que el SFT, y el modelo tiende a explotar cualquier defecto de la función de recompensa (*reward hacking*). Por eso se aplica como última etapa, sobre un modelo que ya ha pasado por SFT, y no lo utilizaremos en la práctica de este tema.

## **4. ¿Qué técnica utilizar en un dominio específico?**

La elección depende, sobre todo, de los datos de los que se disponga:

| Tengo... | Técnica | El modelo aprende... |
|---|---|---|
| Mucho texto del dominio, sin etiquetar | Preentrenamiento autorregresivo continuado | El lenguaje y el conocimiento del dominio |
| Ejemplos de entrada y salida deseada | SFT | La tarea y la forma de responder |
| Una forma de puntuar respuestas | RL | A optimizar ese criterio |

Las tres se pueden encadenar, en ese orden. En la mayoría de los proyectos reales, sin embargo, basta con un **SFT sobre un modelo *instruct* ya existente**, y casi siempre mediante técnicas eficientes en parámetros (PEFT), que son el objeto del siguiente notebook.

Conviene recordar, además, que el ajuste fino cambia el *comportamiento* del modelo, pero no es la mejor herramienta para darle acceso a información que cambia con frecuencia: para eso suele ser preferible recuperar los documentos e incluirlos en el contexto (RAG).

### **Materiales**

En este enlace [notebooks y datasets](https://drive.google.com/file/d/1Ja2jp43LiCMZLHNkTXun7GdnWD9Thy1N/view?usp=sharing) vas a encontrar notebooks y datasets para el entrenamiento de modelos de lenguaje en dominios específicos: un ejemplo de entrenamiento autorregresivo, otro de SFT y el generador de series de la práctica.

### **Práctica en el aula**

Vamos a entrenar un modelo de lenguaje en un dominio específico utilizando el ajuste fino supervisado (SFT). Para ello, utilizaremos el mismo modelo Mistral que acabamos de ver. En este caso, entrenaremos el modelo mediante SFT en el conjunto de datos <code>list_of_strings.jsonl</code>, que contiene ejemplos del tipo:

```
{"text": "<s>[INST] 12 es igual a [/INST] ocho más uno más uno más uno más uno </s>"}
```

Utiliza el notebook <code>serie_generator.ipynb</code> para generar nuevas líneas y medir el error que se produce en el texto generado por el modelo.

### **Referencias**

- Gururangan et al. (2020). [Don't Stop Pretraining: Adapt Language Models to Domains and Tasks](https://arxiv.org/abs/2004.10964)
- Ouyang et al. (2022). [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155) (InstructGPT, RLHF)
- Schulman et al. (2017). [Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347)
- Rafailov et al. (2023). [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290)
- DeepSeek-AI (2025). [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948)